# DS02 Transformer：健康校准与双机制自监督适应

**完整独立代码，按单元执行，不依赖之前的 health_calibration_transformer.py。**
本 notebook 只研究 N-CMAPSS DS02-006。原脚本已经有 P_ssl/P_fused 自监督，本版保留并改进。
本版按用户要求不代运行训练，Notebook 不含执行输出。代码由用户本地运行验证。
默认 train 为单种子正式预算；quick 为短程排错；full 为5个固定种子。
结构与损失改动是待验证方案，不保证融合优于基线。

方法对应：Le Xuan et al. (2024), Eq.23–25 的同机相对 RUL 自监督；
Nejjar et al. (2024), Fig.3–4 / Eq.5–7 的分阶段 hard/soft 域对齐。
这是 Transformer 上的适配与融合，不是两篇原文完整复现；原文分别主要使用 C-MAPSS 与 N-CMAPSS DS03。

原始基线 B0 保留；增强模型使用轻量局部卷积+Transformer+可学习汇聚。
粒子滤波后验均值进入健康分支，后验标准差仅作门控可靠性提示，不等于经过校准的 RUL 不确定性。
Oracle 使用真实健康参数，结构与 PF 模型相同，只是特权信息参考，不能称为数学理论上限。

In [ ]:
# 1. 环境：所有函数和模型都在本文件中，修改 DATA_PATH 后可从上到下运行。
from __future__ import annotations
import copy, csv, hashlib, json, math, platform, random, time
from pathlib import Path
from dataclasses import dataclass, asdict
import h5py
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
from matplotlib import font_manager
from IPython.display import display, Markdown

font_set = {f.name for f in font_manager.fontManager.ttflist}
plt.rcParams['font.family'] = next((x for x in ['Microsoft YaHei','SimHei','Noto Sans CJK SC'] if x in font_set),'DejaVu Sans')
plt.rcParams['axes.unicode_minus'] = False
print('Python:',platform.python_version(),'Torch:',torch.__version__,'CUDA:',torch.cuda.is_available())

## 2. 固定协议与运行预算
源训练：2/5/10/16/18；源验证：20；目标14的前20循环无标签适应，后缀只在所有模型锁定后读取。
前缀不读取 Y/T/hs，后缀观测只进行顺序滤波，不更新模型权重。
50点窗口限定在一个循环中，端点稀疏抽取；这是新的共同协议，不能与历史全窗口 RMSE 直接排名。
等距端点是离线评测采样约定；在线部署需改用事先固定的时间采样表，并为全部模型重建同一协议。
只使用循环计数差；不截顶 RUL、不用随机相邻 batch 做单调损失。

In [ ]:
RUN_MODE = 'train'  # 'quick' / 'train' / 'full'；第一次建议自行改为quick检查环境
DATA_PATH = r'E:/RUL/N-CMAPSS_DS02-006.h5'
RESULT_BASE = Path(DATA_PATH).parent / 'DS02_SSL_runs'  # 与数据集同目录，固定输出到 E:/RUL

@dataclass
class Config:
    data: str = DATA_PATH
    train_units: tuple = (2,5,10,16,18)
    val_units: tuple = (20,)
    target: int = 14
    prefix_cycles: int = 20
    window: int = 50
    endpoints: int = 12
    fs_hz: float = 1.0
    phase_threshold: float = .5
    phase_lookback: int = 30  # 本实现预设30秒向后平均变化率；不是已验证最优值，也不是原文原样复现
    particles: int = 256
    proxy_epochs: int = 60
    pretrain_epochs: int = 80
    adapt_epochs: int = 30
    patience: int = 12
    steps_per_epoch: int = 80
    batch: int = 128
    lr: float = 3e-4
    ssl_weight: float = .2
    domain_weight: float = .03
    phase_weight: float = .1
    covariance_shrink: float = .2
    likelihood_temperature: float = 1.0
    seed: int = 42
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    mode: str = RUN_MODE

def config_for(seed):
    c=Config(seed=seed,data=DATA_PATH,mode=RUN_MODE)
    if RUN_MODE == 'quick':
        c.endpoints=4; c.particles=48; c.proxy_epochs=3
        c.pretrain_epochs=3; c.adapt_epochs=2; c.steps_per_epoch=8; c.batch=32
    if RUN_MODE not in ['quick','train','full']:
        raise ValueError('Unknown RUN_MODE')
    return c

SEEDS = [42,43,44,45,46] if RUN_MODE == 'full' else [42]
assert Path(DATA_PATH).is_file(), DATA_PATH
print(asdict(config_for(SEEDS[0])))
if RUN_MODE=='quick': print('QUICK：真实数据短程检查，不能用本次分数证明精度优势。')

In [ ]:
# 3. 日志与复现工具：每次运行创建新目录，不覆盖原结果。
def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False

def save_json(path,value):
    Path(path).write_text(json.dumps(value,ensure_ascii=False,indent=2),encoding='utf-8')

def names(f,key):
    return [v.decode() if isinstance(v,bytes) else str(v) for v in f[key][:]]

def subset(d,mask): return {k:v[mask] for k,v in d.items()}

def observable(d):
    # 构造滤波器/目标自监督的输入边界：无 Y/T/hs。
    return {k:d[k] for k in ['x','unit','cycle','row','phase','fc']}

def data_fingerprint(path):
    # 快速文件指纹，不冒充全文件 SHA256；大小、mtime 一并保存。
    p=Path(path)
    with p.open('rb') as f:
        first=f.read(2**20); f.seek(max(0,p.stat().st_size-2**20)); last=f.read(2**20)
    return dict(bytes=p.stat().st_size,mtime_ns=p.stat().st_mtime_ns,
                first_last_1MiB_sha256=hashlib.sha256(first+last).hexdigest())

## 4. 读取真实窗口，保留循环与工况标签
飞行阶段为规则生成的实验标签：上升0、平飞1、下降2，尚未验证阶段识别准确率。
默认以过去30秒的平均高度变化率降低原始差分抖动；仍需在源域核查，不能靠测试RUL调阈值。
Fc不作为输入，不混同飞行阶段。30秒平滑和稀疏采样区别于论文预处理，必须披露。
每个窗口末端才读取对应标签；真实健康参数只有源训练和独立 oracle 分支可以使用。
源数据归一化、健康参数尺度、RUL尺度全部仅从训练发动机计算。

In [ ]:
def load_windows(c,split,units,with_y=False,with_theta=False,region='all'):
    result={k:[] for k in ['x','unit','cycle','row','phase','fc']}
    if with_y: result['y']=[]
    if with_theta: result['theta']=[]
    with h5py.File(c.data,'r') as f:
        assert names(f,'A_var')[:3]==['unit','cycle','Fc']
        ai=names(f,'W_var').index('alt')
        a=f[f'A_{split}'][:,:3]  # 故意不读取 hs
        starts=np.r_[0,np.flatnonzero(np.any(a[1:,:2]!=a[:-1,:2],axis=1))+1]
        ends=np.r_[starts[1:],len(a)]
        for s,e in zip(starts,ends):
            u,cy,fc=a[s].astype(int)
            if u not in units or e-s<c.window: continue
            if region=='prefix' and cy>c.prefix_cycles: continue
            if region=='suffix' and cy<=c.prefix_cycles: continue
            points=np.unique(np.linspace(s+c.window-1,e-1,c.endpoints).astype(int))
            for end in points:
                sl=slice(end-c.window+1,end+1)
                w=f[f'W_{split}'][sl].astype('float32')
                xs=f[f'X_s_{split}'][sl].astype('float32')
                lag=min(c.phase_lookback,c.window-1)
                slope=(w[-1,ai]-w[-1-lag,ai])*c.fs_hz/lag
                ph=0 if slope>=c.phase_threshold else (2 if slope<=-c.phase_threshold else 1)
                result['x'].append(np.concatenate([w,xs],axis=-1))
                for key,v in [('unit',u),('cycle',cy),('row',end),('phase',ph),('fc',fc)]: result[key].append(v)
                if with_y: result['y'].append(float(np.asarray(f[f'Y_{split}'][end]).reshape(-1)[0]))
                if with_theta: result['theta'].append(f[f'T_{split}'][end].astype('float32'))
    if not result['x']: raise ValueError(f'No windows {split} {units} {region}')
    d={k:np.asarray(v) for k,v in result.items()}
    assert np.isfinite(d['x']).all()
    return d

def fit_scales(source):
    raw=source['x'][:,-1]
    active=np.flatnonzero(np.ptp(source['theta'],axis=0)>1e-6)
    if not len(active): raise ValueError('训练健康参数没有变化')
    th=source['theta'][:,active]
    # 去除完全线性冗余观测：只用源训练观测识别，原始Transformer仍保留18维。
    sensors=raw[:,4:]; keep=[]; dropped=[]
    for j in range(sensors.shape[1]):
        redundant=False
        for k in keep:
            design=np.c_[sensors[:,k],np.ones(len(sensors))]
            coef=np.linalg.lstsq(design,sensors[:,j],rcond=None)[0]
            relative=np.std(sensors[:,j]-design@coef)/max(np.std(sensors[:,j]),1e-8)
            if relative<1e-5:
                dropped.append([j,k,float(relative)]); redundant=True; break
        if not redundant: keep.append(j)
    return dict(active=active,obs_idx=np.array(keep),dropped=dropped,
                raw_mean=raw.mean(0),raw_std=np.maximum(raw.std(0),1e-5),
                xmin=source['x'].min((0,1)),xmax=source['x'].max((0,1)),
                theta_center=(th.min(0)+th.max(0))/2,theta_scale=np.maximum(np.ptp(th,axis=0),1e-3),
                y_scale=float(max(source['y'].max(),1.)))

## 5. 前向代理模型与留一发动机交叉拟合
没有可调用 CMAPSS 仿真器，因此用源训练的 (W,真实θ) 学习 Xs。
每台源发动机的 PF 特征由排除该发动机的代理模型产生，减轻源域过于理想的反演特征。
用源训练的 out-of-fold 预测残差估计分阶段观测协方差，加入收缩和噪声下限保证正定。
标准化尺度与协方差汇总来自全部源训练发动机，属于训练侧统计；并非对所有预处理做嵌套交叉拟合。
这里协方差包含代理误差、偏差与观测噪声，不能直接解释成真实传感器噪声协方差。

In [ ]:
class ForwardProxy(nn.Module):
    def __init__(self,nhealth,nobs):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(4+nhealth,64),nn.SiLU(),nn.Linear(64,64),nn.SiLU(),nn.Linear(64,nobs))
    def forward(self,w,theta): return self.net(torch.cat([w,theta],dim=-1))

def proxy_arrays(d,sc):
    raw=(d['x'][:,-1]-sc['raw_mean'])/sc['raw_std']
    theta=(d['theta'][:,sc['active']]-sc['theta_center'])/sc['theta_scale']
    return raw[:,:4],theta,raw[:,4:][:,sc['obs_idx']]

def train_proxy(d,sc,c,seed,folder):
    seed_all(seed); w,t,y=proxy_arrays(d,sc)
    w,t,y=[torch.as_tensor(v,dtype=torch.float32,device=c.device) for v in [w,t,y]]
    model=ForwardProxy(t.shape[1],y.shape[1]).to(c.device)
    opt=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4)
    rng=np.random.default_rng(seed); history=[]
    for epoch in range(c.proxy_epochs):
        model.train(); losses=[]
        for ids in np.array_split(rng.permutation(len(w)),max(1,math.ceil(len(w)/256))):
            loss=F.mse_loss(model(w[ids],t[ids]),y[ids])
            opt.zero_grad(); loss.backward(); opt.step(); losses.append(loss.item())
        history.append(dict(epoch=epoch+1,mse=float(np.mean(losses))))
    model.eval()
    folder.mkdir(parents=True,exist_ok=True)
    torch.save(model.state_dict(),folder/'proxy.pt'); pd.DataFrame(history).to_csv(folder/'history.csv',index=False)
    print(folder.name,'final proxy MSE',round(history[-1]['mse'],5),flush=True)
    return model

@torch.no_grad()
def proxy_residual(model,d,sc,c):
    w,t,y=proxy_arrays(d,sc); parts=[]
    for s in range(0,len(w),512):
        parts.append(model(torch.tensor(w[s:s+512],device=c.device),torch.tensor(t[s:s+512],device=c.device)).cpu().numpy())
    return y-np.concatenate(parts)

def observation_covariances(residual,phase,c):
    covs=[]
    for p in range(3):
        r=residual[phase==p]
        if len(r)<max(20,2*residual.shape[1]): r=residual
        # 未中心化二阶矩把代理系统性偏差也计入误差预算，避免过窄后验。
        cov=(r.T@r)/len(r)
        cov=(1-c.covariance_shrink)*cov+c.covariance_shrink*np.diag(np.diag(cov))
        cov+=np.eye(cov.shape[0])*.05**2
        np.linalg.cholesky(cov)
        covs.append(cov)
    return np.asarray(covs)

def fit_prior(d,sc):
    theta=(d['theta'][:,sc['active']]-sc['theta_center'])/sc['theta_scale']
    initial=[]; increments=[]
    for u in np.unique(d['unit']):
        ids=np.flatnonzero(d['unit']==u); cy=np.unique(d['cycle'][ids])
        states=np.array([theta[ids[np.flatnonzero(d['cycle'][ids]==x)[0]]] for x in cy])
        initial.append(states[0])
        increments.extend(np.diff(states,axis=0)/np.diff(cy)[:,None])
    initial=np.asarray(initial); increments=np.asarray(increments)
    return dict(mean=initial.mean(0),std=np.maximum(initial.std(0),.1),
                drift=increments.mean(0),process_std=np.maximum(increments.std(0),.004),
                lower=theta.min(0)-.5,upper=theta.max(0)+.5)

def fit_calibrator(source,sc,c,folder):
    residual=np.zeros((len(source['x']),len(sc['obs_idx'])),dtype='float32')
    folds={}
    for u in np.unique(source['unit']):
        mask=source['unit']==u; training=subset(source,~mask); held=subset(source,mask)
        model=train_proxy(training,sc,c,c.seed+int(u),folder/f'proxy_without_U{u}')
        residual[mask]=proxy_residual(model,held,sc,c)
        folds[int(u)]=dict(model=model,prior=fit_prior(training,sc))
    cov=observation_covariances(residual,source['phase'],c)
    full=train_proxy(source,sc,c,c.seed,folder/'proxy_all_source')
    bundle=dict(model=full,prior=fit_prior(source,sc),cov=cov,scales=sc)
    for pack in folds.values(): pack.update(cov=cov,scales=sc)
    np.savez_compressed(folder/'observation_model.npz',cov=cov,oof_residual=residual,**{k:v for k,v in sc.items() if k!='dropped'})
    save_json(folder/'prior.json',{k:v.tolist() for k,v in bundle['prior'].items()})
    return bundle,folds

## 6. 因果粒子滤波：可保存/续接前缀状态
只在循环变化时传播健康状态；每个选定观测端点更新一次似然。
使用分工况的完整收缩协方差，自动去除近似完全冗余观测，避免简单重复计数。
初始/转移/边界均由源训练估计，是经验先验，不是精确物理规律；允许轻微参数反向变化。
ESS低时系统重采样。输出均值、标准差、ESS和边界质量；不回溯平滑、不读取目标真值。
前缀适应完成后可以冻结网络，并从保存的PF状态继续处理后缀。

In [ ]:
def systematic_resample(weights,rng):
    cdf=np.cumsum(weights); cdf[-1]=1.
    return np.searchsorted(cdf,(rng.random()+np.arange(len(weights)))/len(weights))

@torch.no_grad()
def particle_filter(d,pack,c,states=None):
    assert 'theta' not in d and 'y' not in d and 'hs' not in d
    sc=pack['scales']; prior=pack['prior']; model=pack['model']; model.eval()
    n=c.particles; dim=len(sc['active']); states=copy.deepcopy(states or {})
    result=dict(mean=np.zeros((len(d['x']),dim),dtype='float32'),std=np.zeros((len(d['x']),dim),dtype='float32'),
                ess=np.zeros(len(d['x'])),boundary_mass=np.zeros(len(d['x'])))
    chol=[np.linalg.cholesky(v) for v in pack['cov']]
    invchol=[np.linalg.inv(v) for v in chol]
    for u in np.unique(d['unit']):
        ids=np.flatnonzero(d['unit']==u); ids=ids[np.argsort(d['row'][ids])]
        rng=np.random.default_rng(c.seed+int(u)*1009)
        if int(u) in states:
            state=states[int(u)]; particles=state['particles'].copy(); weights=state['weights'].copy()
            previous=state['cycle']; last_row=state['last_row']; rng.bit_generator.state=state['rng']
            assert d['row'][ids[0]]>last_row, '续接数据必须严格晚于已处理观测'
        else:
            particles=np.clip(rng.normal(prior['mean'],prior['std'],(n,dim)),prior['lower'],prior['upper'])
            weights=np.full(n,1/n); previous=int(d['cycle'][ids[0]])
        for idx in ids:
            delta=int(d['cycle'][idx])-previous; assert delta>=0
            if delta:
                particles+=prior['drift']*delta+rng.normal(size=particles.shape)*prior['process_std']*math.sqrt(delta)
                particles=np.clip(particles,prior['lower'],prior['upper'])
            z=(d['x'][idx,-1]-sc['raw_mean'])/sc['raw_std']
            w=torch.tensor(np.repeat(z[None,:4],n,axis=0),device=c.device,dtype=torch.float32)
            theta=torch.tensor(particles,device=c.device,dtype=torch.float32)
            expected=model(w,theta).cpu().numpy()
            err=z[4:][sc['obs_idx']]-expected
            whitened=err@invchol[int(d['phase'][idx])].T
            logw=np.log(weights+1e-300)-.5*np.sum(whitened**2,axis=1)/c.likelihood_temperature
            logw-=logw.max(); weights=np.exp(logw); weights/=weights.sum()
            mu=weights@particles; sd=np.sqrt(weights@((particles-mu)**2))
            result['mean'][idx]=mu*sc['theta_scale']+sc['theta_center']
            result['std'][idx]=sd*sc['theta_scale']
            result['ess'][idx]=1/np.sum(weights**2)
            near=np.any((particles<=prior['lower']+1e-5)|(particles>=prior['upper']-1e-5),axis=1)
            result['boundary_mass'][idx]=weights[near].sum()
            if result['ess'][idx]<n/2:
                particles=particles[systematic_resample(weights,rng)]; weights.fill(1/n)
            previous=int(d['cycle'][idx])
        states[int(u)]=dict(particles=particles,weights=weights,cycle=previous,
                            last_row=int(d['row'][ids[-1]]),rng=copy.deepcopy(rng.bit_generator.state))
    return result,states

def calibrate_source(source,folds,c):
    combined={}
    for u in np.unique(source['unit']):
        mask=source['unit']==u
        pf,_=particle_filter(observable(subset(source,mask)),folds[int(u)],c)
        for k,v in pf.items():
            if k not in combined: combined[k]=np.zeros((len(source['x']),)+v.shape[1:],dtype=v.dtype)
            combined[k][mask]=v
    return combined

## 7. 原始基线与增强结构（同一训练协议）
B0：原始输入映射、2层/2头/FF64、最后token回归。
增强：3/7核深度卷积提供窗口内局部特征；Transformer后将末端与注意力汇聚融合。
健康分支：投影估计θ，以学习门控融合；PF标准差越大，健康分支的启发式权重越小。
Oracle与估计θ模型使用相同增强结构；另外保留 enhanced_raw，分离结构变化与健康信息贡献。
所有预测窗口都只包含当前末端及过去观测；Transformer的窗口内双向注意力不读取末端之后的数据。

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self,dim,length):
        super().__init__(); pe=torch.zeros(length,dim); p=torch.arange(length).float()[:,None]
        div=torch.exp(torch.arange(0,dim,2).float()*(-math.log(10000.)/dim))
        pe[:,0::2]=torch.sin(p*div); pe[:,1::2]=torch.cos(p*div)
        self.register_buffer('pe',pe[None])
    def forward(self,x): return x+self.pe[:,:x.shape[1]]

class HealthTransformer(nn.Module):
    def __init__(self,window,nhealth,enhanced=True,use_health=True):
        super().__init__(); self.enhanced=enhanced; self.use_health=use_health
        self.input_projection=nn.Sequential(nn.Linear(18,32),nn.LayerNorm(32))
        self.pos_encoder=PositionalEncoding(32,window)
        layer=nn.TransformerEncoderLayer(32,2,64,.1,activation='gelu',batch_first=True,norm_first=True)
        self.encoder=nn.TransformerEncoder(layer,2,norm=nn.LayerNorm(32))
        if enhanced:
            self.local3=nn.Conv1d(32,32,3,groups=32)
            self.local7=nn.Conv1d(32,32,7,groups=32)
            self.local_norm=nn.LayerNorm(32)
            self.pool_score=nn.Linear(32,1); self.pool_gate=nn.Parameter(torch.zeros(1))
        self.health_projection=nn.Sequential(nn.Linear(nhealth,32),nn.GELU(),nn.Linear(32,32))
        self.health_gate=nn.Linear(64,32)
        self.phase_head=nn.Sequential(nn.Linear(32,32),nn.GELU(),nn.Linear(32,3))
        self.regressor=nn.Sequential(nn.Linear(32,32),nn.GELU(),nn.Dropout(.1),nn.Linear(32,1))
        for m in self.modules():
            if isinstance(m,nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)
    def forward(self,x,health,health_sd):
        z=self.input_projection(x)
        if self.enhanced:
            t=z.transpose(1,2)
            local=.5*(self.local3(F.pad(t,(2,0)))+self.local7(F.pad(t,(6,0))))
            z=self.local_norm(z+F.gelu(local.transpose(1,2)))
        z=self.encoder(self.pos_encoder(z)); pooled=z[:,-1]
        if self.enhanced:
            weights=torch.softmax(self.pool_score(z).squeeze(-1),dim=1)
            pooled=pooled+torch.sigmoid(self.pool_gate)*(torch.einsum('bt,btd->bd',weights,z)-pooled)
        phase_logits=self.phase_head(pooled)  # 工况分类从传感器特征产生，不直接读取真实θ
        if self.use_health:
            h=self.health_projection(health)
            confidence=1/(1+health_sd.mean(-1,keepdim=True))
            gate=torch.sigmoid(self.health_gate(torch.cat([pooled,h],dim=-1)))*confidence
            pooled=pooled+gate*h
        return self.regressor(pooled).squeeze(-1),pooled,phase_logits

## 8. 自监督：同发动机、同工况、不同循环的共享权重双分支
对源域和目标前缀都配对，不需要目标RUL。覆盖不同时间跨度，避免只学习相邻循环的微小差异。
预测输出以训练RUL尺度归一化，因此循环差也要除以同一个尺度。
使用MSE而非论文RMSE、同阶段配对和两阶段训练是本实现的改动，均需在报告中披露。

In [ ]:
def make_pairs(d):
    pairs=[]
    for u in np.unique(d['unit']):
        for p in range(3):
            ids=np.flatnonzero((d['unit']==u)&(d['phase']==p))
            cy=np.unique(d['cycle'][ids])
            # 每循环取该阶段最后一个可用窗口，避免把同循环重复标签当作进展。
            representatives=[ids[np.flatnonzero(d['cycle'][ids]==v)[-1]] for v in cy]
            for i,a in enumerate(representatives):
                for gap in [1,3,7,15]:
                    if i+gap<len(representatives): pairs.append((a,representatives[i+gap]))
    if not pairs: raise ValueError('没有同机同阶段跨循环样本对')
    pairs=np.asarray(pairs,dtype=int)
    assert np.all(d['unit'][pairs[:,0]]==d['unit'][pairs[:,1]])
    assert np.all(d['cycle'][pairs[:,1]]>d['cycle'][pairs[:,0]])
    return pairs

def ssl_loss(pred_earlier,pred_later,delta_cycles,y_scale):
    return F.mse_loss(pred_earlier-pred_later,delta_cycles/y_scale)

## 9. 工况对齐：hard 与 soft 都保留
Hard：规则阶段只进入对应的域判别器。Soft：工况分类器输出三个概率，为三个域判别器加权。
分类器用高度导出的阶段标签训练；目标无需RUL。概率在域权重处detach，防止通过改变路由逃避对齐；
这是相对原文的稳定性改动，分类头仍通过阶段交叉熵学习。
每个阶段的源/目标损失分别归一化，避免源/目标样本量不均；缺乏支持的阶段跳过。
梯度反转仅对特征提取器生效，域判别器仍正常学习分类。

In [ ]:
class Reverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx,x,strength): ctx.strength=strength; return x.view_as(x)
    @staticmethod
    def backward(ctx,grad): return -ctx.strength*grad,None

class OperationAlignment(nn.Module):
    def __init__(self):
        super().__init__()
        self.heads=nn.ModuleList([nn.Sequential(nn.Linear(32,32),nn.ReLU(),nn.Linear(32,1)) for _ in range(3)])
    def forward(self,zs,zt,ps,pt,ls,lt,strength,mode):
        if mode=='hard':
            ws=F.one_hot(ps,3).float(); wt=F.one_hot(pt,3).float()
        else:
            ws=torch.softmax(ls,dim=-1).detach(); wt=torch.softmax(lt,dim=-1).detach()
        losses=[]
        for p,head in enumerate(self.heads):
            a,b=ws[:,p],wt[:,p]
            if a.sum()<2 or b.sum()<2: continue
            src=head(Reverse.apply(zs,strength)).squeeze(-1)
            tgt=head(Reverse.apply(zt,strength)).squeeze(-1)
            sl=F.binary_cross_entropy_with_logits(src,torch.zeros_like(src),reduction='none')
            tl=F.binary_cross_entropy_with_logits(tgt,torch.ones_like(tgt),reduction='none')
            losses.append(.5*((sl*a).sum()/a.sum()+(tl*b).sum()/b.sum()))
        return torch.stack(losses).mean() if losses else zs.sum()*0

In [ ]:
# 10. 特征与批次工具。PF/oracle用相同源训练尺度，PF标准差只在health_sd出现。
def features(d,sc,pf=None,oracle=False):
    span=np.where(sc['xmax']-sc['xmin']<1e-8,1.,sc['xmax']-sc['xmin'])
    h=np.zeros((len(d['x']),len(sc['active'])),dtype='float32'); sd=np.zeros_like(h)
    if oracle: h=(d['theta'][:,sc['active']]-sc['theta_center'])/sc['theta_scale']
    elif pf is not None:
        h=(pf['mean']-sc['theta_center'])/sc['theta_scale']; sd=pf['std']/sc['theta_scale']
    out={k:v for k,v in d.items() if k!='theta'}  # 真实θ不传进普通训练/适应函数
    out['x']=(2*(d['x']-sc['xmin'])/span-1).astype('float32')
    out['h']=h.astype('float32'); out['sd']=sd.astype('float32')
    return out

def tensors(d,ids,c):
    return tuple(torch.as_tensor(d[k][ids],dtype=torch.float32,device=c.device) for k in ['x','h','sd'])

@torch.no_grad()
def predict(model,d,c):
    model.eval(); out=[]; phase=[]
    for i in range(0,len(d['x']),c.batch):
        ids=np.arange(i,min(i+c.batch,len(d['x'])))
        p,_,logits=model(*tensors(d,ids,c)); out.append(p.cpu().numpy()); phase.append(logits.cpu().numpy())
    return np.concatenate(out),np.concatenate(phase)

## 11. 训练循环：源验证选模型、独立随机流、自动保存
各方法使用相同源监督采样流和相同最大优化步数；额外损失的计算量不同，记录时间/参数量。
源监督归一化MSE与自监督归一化MSE保持同一寿命单位。域权重逐渐开启，降低早期不稳定。
未给目标标签作微调，所以不伪称少样本有监督 fine-tuning；P_ssl 是无标签自监督适应对照。
保存初始模型也是候选：如果适应使源验证变差，可以回退到适应前模型。

In [ ]:
def train_stage(model,src,val,sc,c,folder,epochs,target=None,ssl=False,alignment=None):
    folder.mkdir(parents=True,exist_ok=True)
    if target is not None: assert not any(k in target for k in ['y','theta','hs'])
    domain=OperationAlignment().to(c.device)
    params=list(model.parameters())+(list(domain.parameters()) if alignment else [])
    opt=torch.optim.AdamW(params,lr=c.lr,weight_decay=1e-4)
    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,factor=.5,patience=3,min_lr=1e-6)
    source_rng=np.random.default_rng(c.seed); aux_rng=np.random.default_rng(c.seed+77)
    ps=make_pairs(src) if ssl else None
    pt=make_pairs(target) if ssl and target is not None else None
    init_pred,_=predict(model,val,c)
    best=float(np.mean((init_pred-val['y']/sc['y_scale'])**2)); best_state=copy.deepcopy(model.state_dict())
    best_epoch=0; bad=0; logs=[]; start=time.perf_counter()
    for epoch in range(1,epochs+1):
        model.train(); domain.train(); sums=[]
        for step in range(c.steps_per_epoch):
            ids=source_rng.integers(len(src['x']),size=c.batch)
            yp,zs,ls=model(*tensors(src,ids,c))
            y=torch.tensor(src['y'][ids]/sc['y_scale'],dtype=torch.float32,device=c.device)
            supervised=F.mse_loss(yp,y); temporal=yp.sum()*0; da=yp.sum()*0; phase=yp.sum()*0
            if ssl:
                for d,pairs in [(src,ps),(target,pt)]:
                    if d is None: continue
                    ij=pairs[aux_rng.integers(len(pairs),size=max(4,c.batch//2))]
                    early=model(*tensors(d,ij[:,0],c))[0]; late=model(*tensors(d,ij[:,1],c))[0]
                    delta=torch.tensor(d['cycle'][ij[:,1]]-d['cycle'][ij[:,0]],dtype=torch.float32,device=c.device)
                    temporal=temporal+ssl_loss(early,late,delta,sc['y_scale'])
            if alignment:
                ti=aux_rng.integers(len(target['x']),size=c.batch)
                _,zt,lt=model(*tensors(target,ti,c))
                sp=torch.tensor(src['phase'][ids],device=c.device)
                tp=torch.tensor(target['phase'][ti],device=c.device)
                progress=((epoch-1)+step/c.steps_per_epoch)/epochs
                strength=2/(1+math.exp(-10*progress))-1
                da=domain(zs,zt,sp,tp,ls,lt,strength,alignment)
                if alignment=='soft': phase=.5*(F.cross_entropy(ls,sp)+F.cross_entropy(lt,tp))
            total=supervised+c.ssl_weight*temporal+c.domain_weight*da+c.phase_weight*phase
            if not torch.isfinite(total): raise FloatingPointError('Nonfinite loss')
            opt.zero_grad(); total.backward(); nn.utils.clip_grad_norm_(params,1.); opt.step()
            sums.append([supervised.item(),temporal.item(),da.item(),phase.item()])
        pred,_=predict(model,val,c)
        mse=float(np.mean((pred-val['y']/sc['y_scale'])**2)); scheduler.step(mse)
        av=np.mean(sums,axis=0)
        row=dict(epoch=epoch,supervised=float(av[0]),ssl=float(av[1]),domain=float(av[2]),
                 phase=float(av[3]),val_RMSE=math.sqrt(mse)*sc['y_scale'],lr=opt.param_groups[0]['lr'])
        logs.append(row); print(folder.name,row,flush=True)
        if mse<best:
            best=mse; best_epoch=epoch; best_state=copy.deepcopy(model.state_dict()); bad=0
        else:
            bad+=1
            if bad>=c.patience: break
    model.load_state_dict(best_state)
    torch.save(best_state,folder/'best.pt')
    # 从磁盘实际重载验证checkpoint，不依赖内存中保留的模型。
    model.load_state_dict(torch.load(folder/'best.pt',map_location=c.device,weights_only=True))
    pd.DataFrame(logs).to_csv(folder/'history.csv',index=False)
    save_json(folder/'training_info.json',dict(best_epoch=best_epoch,selection='source-validation RMSE',
              best_val_rmse=math.sqrt(best)*sc['y_scale'],elapsed_seconds=time.perf_counter()-start,
              parameters=sum(p.numel() for p in model.parameters()),
              gpu_peak_bytes=int(torch.cuda.max_memory_allocated()) if c.device=='cuda' else None))
    return model

In [ ]:
# 12. 指标：同时导出整体、工况、循环进度分段、健康恢复诊断。
def score_metrics(p,y):
    e=np.asarray(p,dtype='float64')-np.asarray(y,dtype='float64')
    with np.errstate(over='ignore'):
        score=np.expm1(np.where(e<0,-e/13,e/10))
    return dict(RMSE=float(np.sqrt(np.mean(e*e))),MAE=float(np.mean(abs(e))),
                NASA_score_sum=float(score.sum()),NASA_score_mean=float(score.mean()))

def evaluate_saved(models,test,truth,sc,c,root,pf):
    rows=[]; grouped=[]
    for name,model in models.items():
        data=features(test,sc,oracle=True) if name=='O_oracle' else features(test,sc,pf=pf if name.startswith('P_') else None)
        p,logits=predict(model,data,c); p=p*sc['y_scale']
        frame=pd.DataFrame({k:test[k] for k in ['unit','cycle','row','phase']})
        frame['true']=truth; frame['prediction']=p
        frame.to_csv(root/name/'predictions.csv',index=False)
        rows.append(dict(method=name,seed=c.seed,unit=c.target,n=len(p),**score_metrics(p,truth)))
        for ph in np.unique(test['phase']):
            m=test['phase']==ph
            grouped.append(dict(method=name,group=f'phase_{ph}',n=int(m.sum()),**score_metrics(p[m],truth[m])))
        # 寿命归一化进度仅用于事后报告；绝不用于输入、配对或训练选择。
        progress=test['cycle']/(test['cycle']+truth)
        for stage,a,b in [('early',0,1/3),('middle',1/3,2/3),('late',2/3,1.01)]:
            m=(progress>=a)&(progress<b)
            if m.any(): grouped.append(dict(method=name,group=stage,n=int(m.sum()),**score_metrics(p[m],truth[m])))
    pd.DataFrame(rows).to_csv(root/'comparison.csv',index=False)
    pd.DataFrame(grouped).to_csv(root/'group_metrics.csv',index=False)
    health=[]
    for j,k in enumerate(sc['active']):
        error=pf['mean'][:,j]-test['theta'][:,k]
        health.append(dict(theta_index=int(k),RMSE=float(np.sqrt(np.mean(error**2))),
                           MAE=float(np.mean(abs(error))),mean_posterior_sd=float(pf['std'][:,j].mean())))
    pd.DataFrame(health).to_csv(root/'health_recovery.csv',index=False)
    return pd.DataFrame(rows)

## 13. 运行前检查：因果性、差值符号、域梯度和形状
以下合成测试在用户执行此单元时运行，只验证实现，不是研究实验。
真实数据的10组实验另在后面执行；交付时本版尚未运行这些测试。

In [ ]:
def implementation_checks():
    seed_all(42)
    assert ssl_loss(torch.tensor([.8,.5]),torch.tensor([.5,.3]),torch.tensor([3.,2.]),10).item()<1e-12
    z=torch.tensor([2.],requires_grad=True); (Reverse.apply(z,.4)*3).sum().backward()
    assert torch.allclose(z.grad,torch.tensor([-1.2]))
    model=HealthTransformer(50,3).eval()
    out=model(torch.randn(4,50,18),torch.zeros(4,3),torch.ones(4,3))
    assert out[0].shape==(4,) and torch.isfinite(out[0]).all()
    class Known(nn.Module):
        def forward(self,w,t): return t
    c=Config(device='cpu',particles=256)
    sc=dict(active=np.array([0]),raw_mean=np.zeros(18),raw_std=np.ones(18),
            obs_idx=np.array([0]),theta_scale=np.ones(1),theta_center=np.zeros(1))
    prior=dict(mean=np.array([-.1]),std=np.array([.2]),drift=np.array([-.01]),
               process_std=np.array([.004]),lower=np.array([-1.]),upper=np.array([1.]))
    pack=dict(model=Known(),scales=sc,prior=prior,cov=np.array([[[.01]]]*3))
    x=np.zeros((12,50,18),dtype='float32'); truth=-.2-.01*np.arange(12); x[:,-1,4]=truth
    d=dict(x=x,unit=np.ones(12,dtype=int),cycle=np.arange(1,13),row=np.arange(12),phase=np.ones(12,dtype=int),fc=np.ones(12,dtype=int))
    full,_=particle_filter(d,pack,c)
    prefix,state=particle_filter(subset(d,np.arange(12)<6),pack,c)
    suffix,_=particle_filter(subset(d,np.arange(12)>=6),pack,c,state)
    assert np.array_equal(full['mean'][:6],prefix['mean'])
    assert np.array_equal(full['mean'][6:],suffix['mean'])
    assert np.sqrt(np.mean((full['mean'][:,0]-truth)**2))<.05
    try:
        particle_filter(dict(d,theta=truth),pack,c)
        raise RuntimeError('真值隔离断言未生效')
    except AssertionError: pass
    pair=make_pairs(d); assert np.all(d['cycle'][pair[:,1]]>d['cycle'][pair[:,0]])
    print('PASS：前后缀因果等价、PF恢复、真值隔离、时序差符号、同机配对、梯度反转、模型形状。')

implementation_checks()

## 14. 完整实验调度：十组对照，重点分离自监督收益，测试最后读取
B0_raw / A1_enhanced_raw 分离结构收益；O_oracle 提供真值健康信息参考；P_pf 单独检验校准。
A2_ssl / A3_ssl_soft 直接在增强Transformer上做自监督/自监督+软工况对齐，不输入估计θ。
P_ssl / P_hard / P_soft / P_fused 分别检查时序、硬工况、软工况和融合。
正式五种子保存各次结果与均值/标准差，不挑最好seed。目标仍只有一台，不能将seed数当发动机数。

In [ ]:
def run_experiment(c):
    seed_all(c.seed); start=time.perf_counter()
    root=RESULT_BASE/f'{c.mode}_seed{c.seed}_{time.strftime("%Y%m%d_%H%M%S")}_{time.time_ns()%1000000}'
    root.mkdir(parents=True,exist_ok=True); save_json(root/'config.json',asdict(c))
    print('OUTPUT:',root.resolve(),flush=True)
    source=load_windows(c,'dev',c.train_units,True,True)
    validation=load_windows(c,'dev',c.val_units,True,True)  # theta只供oracle和最终验证诊断
    prefix=load_windows(c,'test',[c.target],region='prefix')
    assert set(source['unit']).isdisjoint(set(validation['unit']))
    assert c.target not in set(source['unit'])|set(validation['unit'])
    assert prefix['cycle'].max()<=c.prefix_cycles and 'y' not in prefix and 'theta' not in prefix
    for u in c.train_units:
        mask=source['unit']==u
        if np.ptp(source['y'][mask]+source['cycle'][mask])>1e-5:
            raise ValueError('Y+cycle不是常数，不能使用当前时序自监督公式')
    sc=fit_scales(source)
    with h5py.File(c.data,'r') as f:
        health_names=names(f,'T_var'); sensor_names=names(f,'X_s_var')
    manifest=dict(data=data_fingerprint(c.data),train=list(c.train_units),val=list(c.val_units),
                  target=c.target,prefix_cycles=c.prefix_cycles,
                  active_health=[health_names[i] for i in sc['active']],
                  removed_observations=[dict(removed=sensor_names[j],reference=sensor_names[k],relative_residual=r) for j,k,r in sc['dropped']],
                  observation_channels=[sensor_names[i] for i in sc['obs_idx']],
                  protocol='cycle-local 50-row windows; sparse equally spaced endpoints; no historical dense-score ranking',
                  selection='source validation only; all checkpoints locked before target suffix read',
                  y_scale=sc['y_scale'],versions=dict(torch=torch.__version__,numpy=np.__version__,python=platform.python_version()))
    save_json(root/'manifest.json',manifest)
    print('active theta:',manifest['active_health'],'redundant:',manifest['removed_observations'],flush=True)
    pack,folds=fit_calibrator(source,sc,c,root)
    pf_source=calibrate_source(source,folds,c)
    pf_val,_=particle_filter(observable(validation),pack,c)
    pf_prefix,prefix_state=particle_filter(prefix,pack,c)
    torch.save(prefix_state,root/'pf_prefix_state.pt')
    for n,d,pf in [('source',source,pf_source),('validation',validation,pf_val),('prefix',prefix,pf_prefix)]:
        np.savez_compressed(root/f'pf_{n}.npz',**pf,unit=d['unit'],cycle=d['cycle'],row=d['row'])
    src=features(source,sc,pf_source); va=features(validation,sc,pf_val); tar=features(prefix,sc,pf_prefix)
    sraw=features(source,sc); vraw=features(validation,sc)
    soracle=features(source,sc,oracle=True); voracle=features(validation,sc,oracle=True)
    models={}
    for name,s,v,enh,h in [('B0_raw',sraw,vraw,False,False),('A1_enhanced_raw',sraw,vraw,True,False),
                          ('O_oracle',soracle,voracle,True,True),('P_pf',src,va,True,True)]:
        seed_all(c.seed)
        model=HealthTransformer(c.window,len(sc['active']),enhanced=enh,use_health=h).to(c.device)
        models[name]=train_stage(model,s,v,sc,c,root/(name+'_pretrain'),c.pretrain_epochs)
    common_pf=copy.deepcopy(models['P_pf'])
    common_raw=copy.deepcopy(models['A1_enhanced_raw'])
    # 自监督主线独立于校准：两个分支均不输入真实或估计θ。
    raw_target=features(prefix,sc)
    for name,align in [('A2_ssl',None),('A3_ssl_soft','soft')]:
        seed_all(c.seed)
        models[name]=train_stage(copy.deepcopy(common_raw),sraw,vraw,sc,c,root/name,c.adapt_epochs,
                                 target=raw_target,ssl=True,alignment=align)
    for name,use_ssl,align in [('P_pf',False,None),('P_ssl',True,None),('P_hard',False,'hard'),
                              ('P_soft',False,'soft'),('P_fused',True,'soft')]:
        seed_all(c.seed)
        models[name]=train_stage(copy.deepcopy(common_pf),src,va,sc,c,root/name,c.adapt_epochs,
                                 target=tar if use_ssl or align else None,ssl=use_ssl,alignment=align)
    for name,s,v in [('B0_raw',sraw,vraw),('A1_enhanced_raw',sraw,vraw),('O_oracle',soracle,voracle)]:
        seed_all(c.seed)
        models[name]=train_stage(models[name],s,v,sc,c,root/name,c.adapt_epochs)
    # 所有checkpoint已锁定。从这里才读取目标后缀（包括最终评估真值）。
    suffix=load_windows(c,'test',[c.target],True,True,region='suffix')
    assert suffix['cycle'].min()>c.prefix_cycles
    pf_suffix,final_state=particle_filter(observable(suffix),pack,c,prefix_state)
    np.savez_compressed(root/'pf_suffix.npz',**pf_suffix,unit=suffix['unit'],cycle=suffix['cycle'],
                        row=suffix['row'],true_theta=suffix['theta'][:,sc['active']])
    torch.save(final_state,root/'pf_final_state.pt')
    report=evaluate_saved(models,suffix,suffix['y'],sc,c,root,pf_suffix)
    save_json(root/'status.json',dict(completed=True,mode=c.mode,elapsed_seconds=time.perf_counter()-start,
                                     claim='Implementation check only' if c.mode=='quick' else 'Single-seed results; aggregate fixed seeds before conclusions'))
    return root,report

In [ ]:
# 15. 用户本地执行入口。运行本单元将开始读取H5并训练；交付时没有执行输出。
# quick 用真实H5做短程排错；train/full 用配置的正式预算。所有数值都由本次运行产生。
RUN_DIRS=[]; reports=[]
for seed in SEEDS:
    directory,report=run_experiment(config_for(seed))
    RUN_DIRS.append(directory); reports.append(report)
all_results=pd.concat(reports,ignore_index=True)
display(all_results)
aggregate=all_results.groupby('method')[['RMSE','MAE','NASA_score_mean']].agg(['mean','std','count'])
aggregate.to_csv(RUN_DIRS[0]/'all_seed_summary.csv')
all_results.to_csv(RUN_DIRS[0]/'all_seed_results.csv',index=False)
if len(SEEDS)>1: display(aggregate)
else: print('单种子不报告跨种子标准差。')

## 16. 可视化：训练曲线、目标轨迹、校准诊断
图中明确标记运行模式；quick 曲线只证明流程可以运行。不要把随机短训模型的排序当成论文结果。
PF标准差未做覆盖率校准，因此健康轨迹的±2σ只标作后验离散度，不叫95%置信区间。

In [ ]:
root=RUN_DIRS[0]; conf=json.loads((root/'config.json').read_text(encoding='utf-8'))
manifest=json.loads((root/'manifest.json').read_text(encoding='utf-8'))
fig,axs=plt.subplots(1,2,figsize=(13,4.5))
for name in ['B0_raw','A1_enhanced_raw','O_oracle','P_pf']:
    h=pd.read_csv(root/(name+'_pretrain')/'history.csv')
    axs[0].plot(h.epoch,h.val_RMSE,label=name)
for name in ['A1_enhanced_raw','A2_ssl','A3_ssl_soft','P_pf','P_fused']:
    h=pd.read_csv(root/name/'history.csv')
    axs[1].plot(h.epoch,h.val_RMSE,label=name)
for ax in axs:
    ax.set(xlabel='训练epoch',ylabel='源验证RMSE（cycle）'); ax.legend(fontsize=8); ax.grid(alpha=.2)
axs[0].set_title('预训练'); axs[1].set_title('适应阶段（源验证选择）')
fig.suptitle(f'{RUN_MODE} / DS02 / seed={conf["seed"]}：短程模式不用于精度结论')
fig.tight_layout(); fig.savefig(root/'training_curves.png',dpi=150,bbox_inches='tight'); plt.show()

fig,ax=plt.subplots(figsize=(12,4.5))
for name in ['B0_raw','A2_ssl','A3_ssl_soft','O_oracle','P_fused']:
    frame=pd.read_csv(root/name/'predictions.csv').groupby('cycle')[['true','prediction']].mean()
    ax.plot(frame.index,frame.prediction,label=name)
ax.plot(frame.index,frame.true,'k--',label='真实RUL')
ax.set(xlabel='目标发动机飞行循环编号',ylabel='每循环平均RUL（cycle）',
       title=f'Unit {conf["target"]} 后缀预测 / {RUN_MODE} / 每循环选定窗口均值'); ax.legend(); ax.grid(alpha=.2)
fig.tight_layout(); fig.savefig(root/'target_predictions.png',dpi=150,bbox_inches='tight'); plt.show()

pf=np.load(root/'pf_suffix.npz'); hn=manifest['active_health']
fig,axs=plt.subplots(len(hn)+1,1,figsize=(12,2.7*(len(hn)+1)),sharex=True)
for j,n in enumerate(hn):
    frame=pd.DataFrame({'cycle':pf['cycle'],'true':pf['true_theta'][:,j],
                        'mean':pf['mean'][:,j],'std':pf['std'][:,j]}).groupby('cycle').last()
    axs[j].plot(frame.index,frame.true,'k--',label='真实θ（仅最终诊断）')
    axs[j].plot(frame.index,frame['mean'],label='PF后验均值')
    axs[j].fill_between(frame.index,frame['mean']-2*frame['std'],frame['mean']+2*frame['std'],alpha=.2,label='±2σ离散度，未校准覆盖率')
    axs[j].set(title=n,ylabel='无量纲修正量'); axs[j].legend(fontsize=8)
diag=pd.DataFrame({'cycle':pf['cycle'],'ess':pf['ess']/conf['particles']}).groupby('cycle').mean()
axs[-1].plot(diag.index,diag.ess,label='ESS / 粒子数'); axs[-1].axhline(.5,ls='--',color='gray',label='重采样阈值')
axs[-1].set(xlabel='飞行循环编号',ylabel='平均有效粒子比例'); axs[-1].legend()
fig.suptitle(f'校准诊断 / {RUN_MODE} / 每循环最后观测的θ、每循环平均ESS',y=1.01)
fig.tight_layout(); fig.savefig(root/'health_calibration.png',dpi=150,bbox_inches='tight'); plt.show()
display(pd.read_csv(root/'health_recovery.csv'))
print('当前后缀最大边界质量：',float(pf['boundary_mass'].max()))

## 17. 如何运行正式实验与报告结论
1. 修改配置为 `RUN_MODE='train'` 做一个种子完整训练；不根据测试结果调整权重。
2. 固定设置后使用 `full` 运行42–46五个种子；报告全部结果、均值/标准差，不能挑最好一次。
3. 若代理模型/θ反演效果差，先检查工况外推、残差协方差、参数可辨识性和状态先验，不能直接归咎Transformer。
4. 优先比较 A1−B0（结构）、A2−A1（自监督）、A3−A2（软工况对齐）；再比较P_pf−A1（校准）和P_fused。
5. O_oracle 是同增强结构的特权信息参考。其 finite-budget 结果未必最好，不是校准方法严格数学上限。
6. 五seed仍只有一台目标发动机；本协议是前缀适应/后缀预测，不是独立目标发动机泛化。

**未声称完成的部分**：真正CMAPSS物理模型反演、完整论文逐项复现、已证明精度提升、校准过的RUL预测区间。
参考：[数据集](https://doi.org/10.3390/data6010005)、[时序自监督](https://doi.org/10.1016/j.ress.2024.110296)、
[工况对齐](https://doi.org/10.1016/j.ress.2023.109718)。